In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
from scipy import stats

In [4]:
mlb_pitchers = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')
mlb_pitchers

,Player,Debut Year,Debut Age,Retirement Year,Retirement Age,Career Length,Wins,Losses,Win Percentage,Total Decisions,...,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Hall of Fame
0,Cy Young,1890,23,1911,44,21,511,315,0.619,826,...,29565,138,2.84,1.130,8.7,0.2,1.5,3.4,2.30,1
1,Pud Galvin,1875,18,1892,35,17,365,310,0.541,675,...,25415,107,2.96,1.191,9.6,0.2,1.1,2.7,2.43,1
2,Walter Johnson,1907,19,1927,39,20,417,279,0.599,696,...,23415,147,2.38,1.061,7.5,0.1,2.1,5.3,2.57,1
3,Phil Niekro,1964,25,1987,48,23,318,274,0.537,592,...,22677,115,3.62,1.268,8.4,0.8,3.0,5.6,1.85,1
4,Nolan Ryan,1966,19,1993,46,27,324,292,0.526,616,...,22575,112,2.97,1.247,6.6,0.5,4.7,9.5,2.04,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,1919,23,1928,32,9,49,80,0.380,129,...,4453,90,3.89,1.518,10.3,0.3,3.4,2.8,0.82,0
1211,Sammy Ellis,1962,21,1969,28,7,63,58,0.521,121,...,4296,88,3.90,1.340,8.7,1.1,3.4,6.1,1.79,0
1212,Lil Stoner,1922,23,1931,32,9,50,57,0.467,107,...,4466,87,4.13,1.548,10.6,0.6,3.4,2.7,0.80,0
1213,Johnny Humphries,1938,23,1946,31,8,52,63,0.452,115,...,4342,97,3.80,1.394,9.2,0.4,3.4,2.8,0.85,0


# K-Fold and GridSearch

In [6]:
gbc = GradientBoostingClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gbc.fit(X_train, y_train)

GradientBoostingClassifier()

In [22]:
%%time

# Define two types of K-Fold Cross Validation strategies so that either one can be used
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Define model
gbc = GradientBoostingClassifier()

param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.6, 0.8, 1.0],
    "max_features": ["sqrt", "log2", None]
}

# Define GridSearchCV with KFold
grid_search = GridSearchCV(
    estimator=gbc,
    param_grid=param_grid,
    cv=skf,
    scoring='recall',
    n_jobs=-1
)

# Fit on your training data
grid_search.fit(X_train, y_train)

# Best model
#best_model = grid_search.best_estimator_
gbc_best = grid_search.best_estimator_
gbc_best.fit(X_train, y_train)

CPU times: user 30.7 s, sys: 16.7 s, total: 47.4 s
Wall time: 28min 17s


GradientBoostingClassifier(learning_rate=0.2, max_depth=2, max_features='sqrt',
                           min_samples_leaf=2, min_samples_split=5,
                           subsample=0.6)

In [23]:
import joblib
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_precision_standard_kfold.sav')
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_precision_stratified_kfold.sav')
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_f1_standard_kfold.sav')
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_f1_stratified_kfold.sav')
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_recall_standard_kfold.sav')
#joblib.dump(gbc_best, 'stored_models/model_gbc_best_recall_stratified_kfold.sav')

['stored_models/model_gbc_best_recall_stratified_kfold.sav']

# Results to Dataframes/CSV files

In [44]:
gbc_best = joblib.load('stored_models/model_gbc_best_recall_stratified_kfold.sav')

In [45]:
# Define the same KFold strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Get out-of-fold predictions using the tuned model
y_pred_all_best = cross_val_predict(gbc_best, X, y, cv=skf)

# Get predicted probabilities
y_proba_all_best = cross_val_predict(gbc_best, X, y, cv=skf, method='predict_proba')

# Regular predictions
y_pred_all_best = cross_val_predict(gbc_best, X, y, cv=skf)

# If binary classification, pick the probability of the predicted class
# (confidence in whichever class was chosen)
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all_best])
confidence_scores = y_proba_all_best[np.arange(len(y_proba_all_best)), y_pred_indices]

# Build results dataframe
gbc_gs_results = pd.DataFrame()
gbc_gs_results['Player'] = mlb_pitchers['Player']
gbc_gs_results['Hall of Fame'] = y
gbc_gs_results['Prediction'] = y_pred_all_best
gbc_gs_results['Prediction Correct'] = (gbc_gs_results['Hall of Fame'] == gbc_gs_results['Prediction']).astype(int)
gbc_gs_results['Prediction Confidence'] = confidence_scores
gbc_gs_results

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.999641
1,Pud Galvin,1,1,1,0.703196
2,Walter Johnson,1,1,1,0.999984
3,Phil Niekro,1,1,1,0.984469
4,Nolan Ryan,1,1,1,0.881014
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.999702
1211,Sammy Ellis,0,0,1,0.999566
1212,Lil Stoner,0,0,1,0.999560
1213,Johnny Humphries,0,0,1,0.999783


In [46]:
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_Precision_Standard_KFold.csv', index=False)
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_Precision_Stratified_KFold.csv', index=False)
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_F1_Standard_KFold.csv', index=False)
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_F1_Stratified_KFold.csv', index=False)
#gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv', index=False)
gbc_gs_results.to_csv('stored_results/Results_GBC_GS_Best_Recall_Stratified_KFold.csv', index=False)

# Old version with no GridSearch or Scaling

In [4]:
gbc = GradientBoostingClassifier()

# Features
X = mlb_pitchers.drop(columns=['Hall of Fame', 'Player'])

# Target
y = mlb_pitchers['Hall of Fame']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)

GradientBoostingClassifier()

In [9]:
from sklearn.model_selection import cross_val_predict

# Generate out-of-fold predictions for all data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_all = cross_val_predict(gbc, X, y, cv=kf)


# Get predicted probabilities
y_proba_all = cross_val_predict(gbc, X, y, cv=kf, method='predict_proba')

# Regular predictions
y_pred_all = cross_val_predict(gbc, X, y, cv=kf)


# If binary classification, pick the probability of the predicted class
# (confidence in whichever class was chosen)
y_pred_indices = np.array([list(np.unique(y)).index(p) for p in y_pred_all])
confidence_scores = y_proba_all[np.arange(len(y_proba_all)), y_pred_indices]

# Build results dataframe
gbc_cv_results = pd.DataFrame()
gbc_cv_results['Player'] = mlb_pitchers['Player']
gbc_cv_results['Hall of Fame'] = y
gbc_cv_results['Prediction'] = y_pred_all
gbc_cv_results['Prediction Correct'] = (gbc_cv_results['Hall of Fame'] == gbc_cv_results['Prediction']).astype(int)
gbc_cv_results['Prediction Confidence'] = confidence_scores

#df['percentage_of_max'] = (df['values'] / max_value) * 100


gbc_cv_results

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.997662
1,Pud Galvin,1,0,0,0.786328
2,Walter Johnson,1,1,1,0.998681
3,Phil Niekro,1,1,1,0.990892
4,Nolan Ryan,1,1,1,0.936866
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.999353
1211,Sammy Ellis,0,0,1,0.999465
1212,Lil Stoner,0,0,1,0.999390
1213,Johnny Humphries,0,0,1,0.999353


In [10]:
gbc_cv_results.to_csv('Results GBC no GS.csv', index=False)